In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import LinearSVC

from sklearn.metrics import confusion_matrix

# load Data

In [8]:

df = pd.read_csv("clean_dataset.csv")

# Clean Column Names

In [9]:

df.columns = df.columns.str.strip()
print(df.columns)
print(df.head())

Index(['date', 'steps', 'calories_burned', 'distance_km', 'active_minutes',
       'sleep_hours', 'heart_rate_avg', 'workout_type', 'weather_conditions',
       'location', 'mood'],
      dtype='str')
         date  steps  calories_burned  distance_km  active_minutes  \
0  2023-01-01   4530          2543.02        16.10             613   
1  2023-01-01  11613          1720.76         8.10             352   
2  2023-01-01  27335          1706.35         3.57             236   
3  2023-01-01  13459          2912.38         6.41            1329   
4  2023-01-01  15378          3344.51        17.88              52   

   sleep_hours  heart_rate_avg workout_type weather_conditions location  \
0          1.5             176      Walking              Clear     Park   
1          6.3             128      Cycling                Fog     Park   
2          6.7             134         Yoga               Snow     Park   
3         11.6             116     Swimming               Rain   Office   
4  

# Remove Unwanted Column

In [10]:
df = df.drop(columns=['date'])


# Create FITNESS

In [11]:




df['fitness_score'] = (
    df['steps'] +
    df['calories_burned'] +
    df['active_minutes']
) / 3

In [12]:

df['fitness'] = pd.qcut(
    df['fitness_score'],
    q=3,
    labels=['Low', 'Medium', 'High']
)


In [13]:
print(df['fitness'].value_counts())

fitness
Low       333334
Medium    333333
High      333333
Name: count, dtype: int64


# Define X and y

In [14]:

y = df['fitness']
X = df.drop(columns=['fitness', 'fitness_score'])


# Encode Target (fitness)

In [15]:
le = LabelEncoder()
y = le.fit_transform(y)

# One-Hot Encode Features

In [16]:


X = pd.get_dummies(X, drop_first=True)

print(X.head())

   steps  calories_burned  distance_km  active_minutes  sleep_hours  \
0   4530          2543.02        16.10             613          1.5   
1  11613          1720.76         8.10             352          6.3   
2  27335          1706.35         3.57             236          6.7   
3  13459          2912.38         6.41            1329         11.6   
4  15378          3344.51        17.88              52          7.4   

   heart_rate_avg  workout_type_Gym Workout  workout_type_Running  \
0             176                     False                 False   
1             128                     False                 False   
2             134                     False                 False   
3             116                     False                 False   
4              84                     False                 False   

   workout_type_Swimming  workout_type_Walking  ...  weather_conditions_Fog  \
0                  False                  True  ...                   False   


# Random Forest

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

In [19]:
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [20]:
y_pred = model.predict(X_test)




In [21]:
print("Accuracy:", accuracy_score(y_test, y_pred))


print(classification_report(y_test, y_pred))

Accuracy: 0.987075
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     66667
           1       0.99      0.99      0.99     66667
           2       0.98      0.98      0.98     66666

    accuracy                           0.99    200000
   macro avg       0.99      0.99      0.99    200000
weighted avg       0.99      0.99      0.99    200000



# Naive bayes

In [22]:
nb_model = GaussianNB()

nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.95804
              precision    recall  f1-score   support

           0       0.97      0.97      0.97     66667
           1       0.97      0.97      0.97     66667
           2       0.94      0.94      0.94     66666

    accuracy                           0.96    200000
   macro avg       0.96      0.96      0.96    200000
weighted avg       0.96      0.96      0.96    200000



# Scale data For SVM

In [23]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_small = X_train[:50000]
y_train_small = y_train[:50000]

# Train SVM

In [24]:
# train SVM
svm_model = LinearSVC()
svm_model.fit(X_train_small, y_train_small)

# predict
y_pred_svm = svm_model.predict(X_test)

# evaluate
print("\n===== SVM RESULTS =====")
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))
print("SVM Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))




===== SVM RESULTS =====
SVM Accuracy: 0.88536
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     66667
           1       0.86      0.94      0.89     66667
           2       0.92      0.72      0.81     66666

    accuracy                           0.89    200000
   macro avg       0.89      0.89      0.88    200000
weighted avg       0.89      0.89      0.88    200000

SVM Confusion Matrix:
 [[66664     0     3]
 [    0 62531  4136]
 [ 8208 10581 47877]]


# Confusion matrices(For all models)

In [25]:
from sklearn.metrics import confusion_matrix

print("Random Forest:\n", confusion_matrix(y_test, y_pred))

print("\nNaive Bayes:\n", confusion_matrix(y_test, y_pred_nb))

print("\nSVM:\n", confusion_matrix(y_test, y_pred_svm))

Random Forest:
 [[65932     0   735]
 [    0 65842   825]
 [  532   493 65641]]

Naive Bayes:
 [[64588     0  2079]
 [    0 64545  2122]
 [ 2063  2128 62475]]

SVM:
 [[66664     0     3]
 [    0 62531  4136]
 [ 8208 10581 47877]]


# Model comparison

In [26]:
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))

Random Forest Accuracy: 0.987075
Naive Bayes Accuracy: 0.95804
SVM Accuracy: 0.88536


# Comparison Table

In [27]:
results = pd.DataFrame({
    "Model": ["Random Forest", "Naive Bayes", "SVM"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_svm)
    ]
})

print(results)

           Model  Accuracy
0  Random Forest  0.987075
1    Naive Bayes  0.958040
2            SVM  0.885360


# Save Result

In [28]:
results.to_csv("model_comparison.csv", index=False)

print("✅ Comparison file saved")

✅ Comparison file saved


# CLassification data

In [30]:
classification_data = pd.DataFrame({
    "Actual": y_test,
    "RandomForest": y_pred,
    "NaiveBayes": y_pred_nb,
    "SVM": y_pred_svm
})

In [31]:
classification_data.to_csv("classification_results.csv", index=False)

print("✅ Classification data saved successfully")

✅ Classification data saved successfully
